# Developmental Mapping of GPR92 (LPAR5) in Human Neonatal Pancreas and Chronic Pancreatitis
Clean, reproducible notebook. Data files are expected under `data/` (not committed to git).


In [ ]:
from __future__ import annotations

from pathlib import Path
import os

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

# Project paths (portable)
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True, parents=True)
PROCESSED_DIR.mkdir(exist_ok=True, parents=True)

# Scanpy defaults
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, frameon=False)

# Local project utilities
from src.preprocessing import load_olaniru_long_matrix_csv, safe_load_omix_txt, intersect_genes_and_concat
from src.scoring import add_gene_set_scores
from src.plotting import umap_multi, dotplot_markers, violin_by_group
from src.utils import fraction_positive


## 1) Load datasets

Place raw files under `data/raw/` and update the filenames below.

In [ ]:
# ---- EDIT FILENAMES AS NEEDED ----
# Olaniru dataset (long matrix .csv.gz)
OLANIRU_LONG_CSV_GZ = RAW_DIR / "Olaniru_long_matrix.csv.gz"  # <-- update

# OMIX Yu fetal pancreas UMI matrices
OMIX_MSTRT_UMI_TXT = RAW_DIR / "OMIX236-20-01" / "UMI.txt"   # <-- update
OMIX_10X_UMI_TXT   = RAW_DIR / "OMIX236-20-02" / "UMI.txt"   # <-- update

# Load
adata_olaniru = load_olaniru_long_matrix_csv(OLANIRU_LONG_CSV_GZ)
adata_olaniru.obs["dataset"] = "Olaniru"

adata_yu_mstrt = safe_load_omix_txt("Yu_mSTRTSeq", OMIX_MSTRT_UMI_TXT, chunk_genes=200)
adata_yu_mstrt.obs["dataset"] = "Yu_mSTRTSeq"

adata_yu_10x   = safe_load_omix_txt("Yu_10x", OMIX_10X_UMI_TXT, chunk_genes=200)
adata_yu_10x.obs["dataset"] = "Yu_10x"

print("Olaniru:", adata_olaniru.shape)
print("Yu mSTRT:", adata_yu_mstrt.shape)
print("Yu 10x:", adata_yu_10x.shape)


## 2) Harmonize genes and build combined object

In [ ]:
adata_all = intersect_genes_and_concat(
    adatas=[adata_olaniru, adata_yu_mstrt, adata_yu_10x],
    labels=["Olaniru", "Yu_mSTRTSeq", "Yu_10x"],
    batch_key="batch_source",
)

# Light filtering (keep permissive; fetal immune can be low)
sc.pp.filter_cells(adata_all, min_counts=300)

adata_all


## 3) Integration / latent space

This notebook uses scVI if installed. If you prefer Scanpy integration, swap this section.

In [ ]:
# Optional: scVI integration
# If scvi-tools isn't installed, install via:
#   pip install scvi-tools torch

try:
    import scvi
except ImportError as e:
    raise ImportError(
        "scvi-tools is required for this section. Install with `pip install scvi-tools torch`."
    ) from e

scvi.model.SCVI.setup_anndata(adata_all, batch_key="batch_source")

model = scvi.model.SCVI(adata_all, n_latent=30)
model.train(
    max_epochs=30,
    batch_size=512,
    train_size=0.9,
    early_stopping=True,
    early_stopping_patience=5,
)

adata_all.obsm["X_scVI"] = model.get_latent_representation()


## 4) Neighborhood graph + UMAP

In [ ]:
sc.pp.neighbors(adata_all, use_rep="X_scVI", n_neighbors=20)
sc.tl.umap(adata_all)

umap_multi(
    adata_all,
    color=["batch_source", "dataset"],
    size=8,
)


## 5) Marker overview and immune subsetting

In [ ]:
markers = {
    "Beta": ["INS", "IAPP", "MAFA", "PCSK1"],
    "Alpha": ["GCG", "TTR", "ARX"],
    "Delta": ["SST", "HHEX"],
    "PP": ["PPY"],
    "Endocrine_prog": ["NEUROG3", "NKX2-2", "NEUROD1", "PAX4"],
    "Ductal": ["KRT19", "KRT8", "KRT18", "MUC1", "SOX9"],
    "Acinar": ["PRSS1", "CPA1", "CTRB2", "REG1A"],
    "Endothelial": ["KDR", "ESAM", "PECAM1", "VWF"],
    "Mesenchymal": ["COL1A1", "COL1A2", "COL3A1", "DCN", "LUM"],
    "Immune": ["PTPRC", "LYZ", "TYROBP", "LST1"],
}

dotplot_markers(adata_all, markers, groupby="batch_source")


In [ ]:
# If you have curated annotations, use them.
# Otherwise, compute a simple immune score to subset immune-like cells.
immune_gene_set = [g for g in ["PTPRC", "LYZ", "TYROBP", "LST1", "HLA-DRA", "HLA-DRB1"] if g in adata_all.var_names]
myeloid_gene_set = [g for g in ["LYZ", "TYROBP", "LST1", "APOE", "C1QA", "C1QB", "C1QC", "MARCO"] if g in adata_all.var_names]

add_gene_set_scores(adata_all, {"pan_immune_score": immune_gene_set, "myeloid_mac_score": myeloid_gene_set})

immune = adata_all[adata_all.obs["pan_immune_score"] > 0.2].copy()
immune.obs_names_make_unique()
print("Immune subset:", immune.shape)

sc.pp.neighbors(immune, use_rep="X_scVI", n_neighbors=20)
sc.tl.umap(immune)

umap_multi(immune, color=["pan_immune_score", "myeloid_mac_score"], size=12)


## 6) GPR92 / LPAR5 expression in immune and myeloid-like cells

In [ ]:
if "LPAR5" in immune.var_names:
    umap_multi(immune, color=["LPAR5", "pan_immune_score", "myeloid_mac_score"], size=15)
    frac = fraction_positive(immune, gene="LPAR5", groupby=None)
    print(f"LPAR5+ fraction in immune subset: {frac:.3f}")
else:
    print("LPAR5 not found in var_names (check gene symbol conventions).")


## 7) Save processed outputs

In [ ]:
# Save AnnData objects (excluded from git by .gitignore)
immune_out = PROCESSED_DIR / "immune_subset.h5ad"
all_out = PROCESSED_DIR / "adata_all_integrated.h5ad"

immune.write_h5ad(immune_out)
adata_all.write_h5ad(all_out)

print("Wrote:", immune_out)
print("Wrote:", all_out)
